# Fase 3.5 — novos dados internos, ajuste fino isolado

Este notebook **não inicia treino ao abrir**. Execute primeiro o piloto do dataset público, confira classes, grupos e espaço, e só então prepare a seleção interna. A fonte Community Forensics–Small é licenciada para pesquisa não comercial; não usar as imagens na futura API comercial sem revisão de licença. RAISE/MNW e as 20 imagens antigas nunca entram no treino, seleção ou calibração. Nenhum resultado externo é usado para escolher parâmetros.

O acervo completo tem cerca de 260 GB; estas células baixam apenas os shards explicitamente escolhidos para `/content`, sem `streaming=True`. A amostra JPEG e seu manifesto ficam no Drive; cada sessão copia a amostra congelada para `/content`. O piloto mostrou imagens suficientes das duas classes e 77 geradores sintéticos elegíveis, mas todas as fotos reais do shard 117 têm `real_source=n/a`. O preparo registra essa limitação e divide essas fotos por imagem, sem alegar validação por fonte real inédita. Execute `prepare` somente após sincronizar um commit que contenha essa correção.

In [4]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

GPU: Tesla T4
Dependências disponíveis.


In [5]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_5', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

Mounted at /content/drive
Drive: /content/drive/MyDrive/TCC_Final | livre reportado: 8.16 GB
/content livre: 207.36 GB


In [30]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

Credencial desbloqueada nesta sessão.


In [31]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase35-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_5_dados.py', 'experimentos/fase3_5_finetune.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc35_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.5 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

Baixando snapshot do GitHub...
Código sincronizado: e9a6d79e65c11223eb5119577afea3f364d41945


In [32]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_5_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

Código: /content/tcc_codigo/e9a6d79e65c11223eb5119577afea3f364d41945


## Dados novos: piloto antes de congelar

O piloto baixa os shards escolhidos por inteiro para `/content` e informa contagens de classes, fontes e geradores. Na execução registrada, o shard 0 trouxe 2.939 sintéticas e o 117 trouxe 2.992 reais, todas com fonte individual `n/a`. A seleção de geradores sintéticos permanece separada por grupo; as fotos reais sem fonte individual usam divisão determinística por imagem e essa limitação é persistida no plano. A preparação ainda recusa amostras insuficientes e duplicatas. Não ajuste os shards usando resultados do benchmark RAISE/MNW.

In [9]:
# 6. PILOTO MANUAL, sem seleção persistida e sem treino.
SHARDS = ['data/HFCF_small_0.parquet', 'data/HFCF_small_117.parquet']
shard_args = [item for shard in SHARDS for item in ('--shard', shard)]
run_live('piloto 3.5', [sys.executable, '-u', str(DATA), 'pilot', '--root', str(ROOT), *shard_args])

piloto 3.5: iniciando
data/HFCF_small_0.parquet: candidatos reais=0 sintéticos=2939
data/HFCF_small_117.parquet: candidatos reais=2992 sintéticos=0
Piloto: {0: 2992, 1: 2939} | grupos reais= 1 | geradores= 78 | bytes= 1384257833 | commit= 6c539a534c07917307c381f5af4053c6091b5278
Classe 0: grupos elegíveis (20+): 1; principais: [('n/a', 2992)]
Classe 1: grupos elegíveis (20+): 77; principais: [('employelalisa/min', 51), ('lewdryuna/a-rainier', 46), ('warriormama777/abyssorangemix', 46), ('1mohitmanoj/surya', 41), ('aacee/textual_inversion_cat', 41), ('aacee/textual_inversion_sksship', 41), ('briannlongzhao/0', 41), ('briannlongzhao/2', 41), ('briannlongzhao/textual_inversion', 41), ('chaitanyakusumanchi/chaitanya_photorealism', 41), ('danbochman/ccxl', 41), ('hangeol/textual_inversion_cat', 41)]
Piloto não cria seleção nem inicia treino.
piloto 3.5: concluído em 0.2 min


In [13]:
# 7. PREPARAÇÃO MANUAL: só após verificar o piloto e a quota do Drive.
# A seleção é fixada antes do treino; não alterar SHARDS após plan.json existir.
run_live('preparação 3.5', [sys.executable, '-u', str(DATA), 'prepare', '--root', str(ROOT), *shard_args])

preparação 3.5: iniciando
data/HFCF_small_0.parquet: candidatos reais=0 sintéticos=2939
data/HFCF_small_117.parquet: candidatos reais=2992 sintéticos=0
Aviso: reais com fonte n/a; divisão por imagem, não por fonte.
Seleção persistida: /content/drive/MyDrive/TCC_Final/dados_fase3_5/fase3_5_novos_v1.incompleto/plan.json
preparação 3.5: ativo há 0.6 min; sem nova saída
preparação 3.5: ativo há 1.1 min; sem nova saída
preparação 3.5: ativo há 1.6 min; sem nova saída
preparação 3.5: ativo há 2.1 min; sem nova saída
preparação 3.5: ativo há 2.6 min; sem nova saída
Hashes OpenFake train: 500/2032
preparação 3.5: ativo há 3.2 min; sem nova saída
preparação 3.5: ativo há 3.7 min; sem nova saída
preparação 3.5: ativo há 4.2 min; sem nova saída
preparação 3.5: ativo há 4.7 min; sem nova saída
Hashes OpenFake train: 1000/2032
preparação 3.5: ativo há 5.6 min; sem nova saída
preparação 3.5: ativo há 6.1 min; sem nova saída
preparação 3.5: ativo há 6.6 min; sem nova saída
preparação 3.5: ativo há 7.

In [14]:
# 8. CÓPIA MANUAL para /content; necessária em cada nova sessão.
run_live('cópia local 3.5', [sys.executable, '-u', str(DATA), 'stage', '--root', str(ROOT)])

cópia local 3.5: iniciando
Cópia/verificação: 100/3000
Cópia/verificação: 200/3000
Cópia/verificação: 300/3000
Cópia/verificação: 400/3000
Cópia/verificação: 500/3000
Cópia/verificação: 600/3000
Cópia/verificação: 700/3000
Cópia/verificação: 800/3000
Cópia/verificação: 900/3000
Cópia/verificação: 1000/3000
Cópia/verificação: 1100/3000
Cópia/verificação: 1200/3000
Cópia/verificação: 1300/3000
Cópia/verificação: 1400/3000
Cópia/verificação: 1500/3000
Cópia/verificação: 1600/3000
Cópia/verificação: 1700/3000
Cópia/verificação: 1800/3000
Cópia/verificação: 1900/3000
Cópia/verificação: 2000/3000
Cópia/verificação: 2100/3000
Cópia/verificação: 2200/3000
Cópia/verificação: 2300/3000
Cópia/verificação: 2400/3000
Cópia/verificação: 2500/3000
Cópia/verificação: 2600/3000
Cópia/verificação: 2700/3000
Cópia/verificação: 2800/3000
Cópia/verificação: 2900/3000
Cópia/verificação: 3000/3000
Dados internos locais prontos: /content/fase3_5_novos_v1
cópia local 3.5: concluído em 0.2 min


In [22]:
# 9. SMOKE MANUAL: hashes, divisões, arquitetura, checkpoint e forward; não treina.
run_live('smoke 3.5', [sys.executable, '-u', str(TRAIN), 'smoke'])

smoke 3.5: iniciando
GPU: Tesla T4
Arquitetura da Fase 2 montada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições 3.5: 3594 719 719 | teste histórico reservado, não avaliado: 1947
Hashes treino: 500/3594
Hashes treino: 1000/3594
Hashes treino: 1500/3594
Hashes treino: 2000/3594
Hashes treino: 2500/3594
Hashes treino: 3000/3594
Hashes treino: 3500/3594
Hashes seleção: 500/719
Hashes calibração: 500/719
Hashes teste reservado: 500/1947
Hashes teste reservado: 1000/1947
Hashes teste reservado: 1500/1947
Pesos iniciais da Fase 3.3 carregados: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt
Smoke OK: /content/drive/MyDrive/TCC_Final/modelos/fase3_5_cfsmall_v1_smoke/smoke.json
smoke 3.5: concluído em 0.5 min


In [23]:
# 10. TREINO MANUAL: inicia ou retoma; não execute antes de validar o smoke.
run_live('ajuste fino 3.5', [sys.executable, '-u', str(TRAIN), 'auto'])

ajuste fino 3.5: iniciando
GPU: Tesla T4
Arquitetura da Fase 2 montada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
Partições 3.5: 3594 719 719 | teste histórico reservado, não avaliado: 1947
Hashes treino: 500/3594
Hashes treino: 1000/3594
Hashes treino: 1500/3594
Hashes treino: 2000/3594
Hashes treino: 2500/3594
Hashes treino: 3000/3594
Hashes treino: 3500/3594
Hashes seleção: 500/719
Hashes calibração: 500/719
Hashes teste reservado: 500/1947
Hashes teste reservado: 1000/1947
Hashes teste reservado: 1500/1947
Pesos iniciais da Fase 3.3 carregados: /content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt
ajuste fino 3.5: ativo há 0.7 min; sem nova saída
[E01/12] lr=3.0e-05 | train_loss=0.4902 | val_loss=0.4149 | AUC=0.8970 | bal_acc=0.8201 | 44s  -> melhor
[E02/12] lr=3.0e-05 | train_loss=0.4747 | val_loss=0.3797 | AUC=0.9196 | bal_acc=0.8294 | 20s  -> melhor
[E03/12] lr=2.9e-05 | train_loss=0.4533 | val_loss=0.360

In [24]:
# 11. Conferência somente-leitura; não reavalia testes externos.
import json
run_dir = ROOT / 'modelos/fase3_5_cfsmall_v1'
for name in ('experiment_config.json', 'split_manifest.json', 'history.csv', 'last.pt', 'best.pt', 'phase3_final.pt', 'internal_metrics.json', 'training_complete.json'):
    path = run_dir / name
    print(name, 'presente' if path.is_file() else 'AUSENTE')
metrics = run_dir / 'internal_metrics.json'
print(json.dumps(json.loads(metrics.read_text(encoding='utf-8')), indent=2) if metrics.is_file() else 'Treino interno ainda não concluído.')

experiment_config.json presente
split_manifest.json presente
history.csv presente
last.pt presente
best.pt presente
phase3_final.pt presente
internal_metrics.json presente
training_complete.json presente
{
  "best_epoch": 10,
  "selection_auc": 0.9487973447871926,
  "calibration_bal_acc_apparent": 0.9212768449824287,
  "temperature": 0.4409470856189728,
  "threshold": 0.44499999999999995,
  "train_count": 3594,
  "selection_count": 719,
  "calibration_count": 719,
  "external_used": false,
  "openfake_test_used": false
}


## Diagnóstico interno após o treino

Execute a célula 12 somente após `training_complete.json` estar presente. Ela usa o checkpoint e o limiar congelados da 3.5, faz inferência apenas na partição interna de seleção e salva um novo relatório em `resultados/diagnostico_interno_fase3_5_v1/`. A calibração, o teste OpenFake e RAISE/MNW permanecem sem nova inferência. Em outra sessão, execute as células 1–5 e 8 antes dela. O script precisa ter sido publicado na branch e sincronizado pela célula 4.

In [29]:
# 12. DIAGNÓSTICO INTERNO MANUAL: somente seleção; não treina.
DIAG = CODE_ROOT / 'experimentos/diagnostico_interno_fase3_5.py'
if not DIAG.is_file():
    raise FileNotFoundError('Sincronize o commit que contém o diagnóstico pela célula 4')
run_live('diagnóstico interno 3.5', [sys.executable, '-u', str(DIAG)])

diagnóstico interno 3.5: iniciando
Partições internas conferidas; teste reservado não será inferido.
GPU: Tesla T4
Checkpoint final carregado | T=0.440947 | limiar=0.445000
validation_selection: 104/719 inferências
validation_selection: 200/719 inferências
validation_selection: 304/719 inferências
validation_selection: 400/719 inferências
validation_selection: 504/719 inferências
validation_selection: 600/719 inferências
validation_selection: 704/719 inferências
validation_selection: 719/719 inferências
validation_selection: acurácia=0.9068 | recall_fake=0.8959 | especificidade_real=0.9200 | AUC=0.9696
Grupos reais com 10+ imagens; até 10 menores taxas de acerto:
  pexels | n=52 | erros=7 | acerto=0.865
  laion | n=48 | erros=6 | acerto=0.875
  cfsmall:n/a | n=225 | erros=13 | acerto=0.942
Grupos sintéticas com 10+ imagens; até 10 menores taxas de acerto:
  sd-3.5 | n=10 | erros=4 | acerto=0.600
  cfsmall:liaobh/textual_inversion_cat | n=19 | erros=6 | acerto=0.684
  cfsmall:roy61/text

## Auditoria dos erros internos

A célula 13 usa apenas `predictions.csv` da seleção interna já inferida. Ela confere checkpoint, partições e rótulos, lê metadados e dimensões das imagens dessa seleção e grava um relatório separado. Não retreina, não recalibra e não acessa OpenFake teste nem benchmarks externos. Em sessão nova, execute 1–5 e 8; depois sincronize o script publicado e rode a célula 13.

In [33]:
# 13. AUDITORIA MANUAL DOS ERROS INTERNOS, sem inferência ou treino.
AUDIT = CODE_ROOT / 'experimentos/auditar_erros_internos_fase3_5.py'
if not AUDIT.is_file():
    raise FileNotFoundError('Sincronize o commit que contém a auditoria pela célula 4')
run_live('auditoria de erros 3.5', [sys.executable, '-u', str(AUDIT)])

auditoria de erros 3.5: iniciando
Metadados internos: 100/719
Metadados internos: 200/719
Metadados internos: 300/719
Metadados internos: 400/719
Metadados internos: 500/719
Metadados internos: 600/719
Metadados internos: 700/719
Metadados internos: 719/719
Seleção: 719 imagens; FP=26; FN=41
dataset:
  CommunityForensics-Small: 44/537 erros (FP=13, FN=31)
  OpenFake: 23/182 erros (FP=13, FN=10)
original_format:
  JPEG: 15/123 erros (FP=13, FN=2)
  PNG: 7/55 erros (FP=0, FN=7)
  WEBP: 1/4 erros (FP=0, FN=1)
  não_registrado: 44/537 erros (FP=13, FN=31)
processed_size_band:
  lado_curto_512_a_1023: 54/490 erros (FP=13, FN=41)
  lado_curto_menor_512: 13/229 erros (FP=13, FN=0)
Erros mais confiantes (probabilidade da classe incorreta):
  FN | CommunityForensics-Small | cfsmall:haya44/shrt-finetune | p_fake=0.020 | 7d13c842733e934591b8d3eeaae9b6d4f027fe844c34a21af7d8ca03becb7af6.jpg
  FN | OpenFake | midjourney-6 | p_fake=0.021 | midjourney-6__00009.jpg
  FN | CommunityForensics-Small | cfs

## Revisão visual direcionada dos erros

A célula 14 usa apenas os CSVs internos já salvos pela célula 13. Mostra uma tabela cruzando dataset, classe e resolução e exibe até seis falsos positivos e seis falsos negativos com maior probabilidade atribuída à classe incorreta. Não altera arquivos nem executa inferência. Examine se há conteúdo, aparência ou defeito de rótulo recorrente antes de propor outro treino.

In [34]:
# 14. VISUALIZAÇÃO INTERNA: leitura dos relatórios da célula 13.
import csv
from collections import defaultdict
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display
audit_dir = ROOT / 'resultados/auditoria_erros_internos_fase3_5_v1'
with (audit_dir / 'selection_metadata.csv').open(newline='', encoding='utf-8') as handle:
    records = list(csv.DictReader(handle))
cross = defaultdict(lambda: [0, 0])
for row in records:
    key = (row['dataset'], 'real' if row['label'] == '0' else 'sintetica', row['processed_size_band'])
    cross[key][0] += 1
    cross[key][1] += bool(row['error_type'])
print('Dataset | classe | lado curto processado | total | erros')
for key, (total, errors) in sorted(cross.items()):
    print(*key, total, errors, sep=' | ')
errors = [row for row in records if row['error_type']]
selected = []
for kind in ('FP', 'FN'):
    selected.extend(sorted((r for r in errors if r['error_type'] == kind),
                           key=lambda r: float(r['wrong_label_probability']), reverse=True)[:6])
sheet = Image.new('RGB', (960, 4 * 290), 'white')
draw = ImageDraw.Draw(sheet)
for index, row in enumerate(selected):
    x, y = (index % 3) * 320, (index // 3) * 290
    with Image.open(row['path']) as opened:
        thumb = ImageOps.contain(ImageOps.exif_transpose(opened).convert('RGB'), (300, 220))
    sheet.paste(thumb, (x + (320-thumb.width)//2, y + 5))
    draw.text((x+8, y+230), f"{row['error_type']} | {row['dataset'][:18]} | p_fake={float(row['probability_fake']):.3f}", fill='black')
    draw.text((x+8, y+249), row['group'][:38], fill='black')
display(sheet)
print('Imagens exibidas:', len(selected), '| somente seleção interna; não usar estes casos para ajustar limiar.')

Dataset | classe | lado curto processado | total | erros
CommunityForensics-Small | real | lado_curto_512_a_1023 | 18 | 1
CommunityForensics-Small | real | lado_curto_menor_512 | 207 | 12
CommunityForensics-Small | sintetica | lado_curto_512_a_1023 | 312 | 31
OpenFake | real | lado_curto_512_a_1023 | 80 | 12
OpenFake | real | lado_curto_menor_512 | 20 | 1
OpenFake | sintetica | lado_curto_512_a_1023 | 80 | 10
OpenFake | sintetica | lado_curto_menor_512 | 2 | 0


Imagens exibidas: 12 | somente seleção interna; não usar estes casos para ajustar limiar.
